# Algerian Dish Classifier — ResNet-50 Transfer Learning

15-class classifier trained on 1500 images (100 per class) via transfer learning.

**Before running:**
1. `Runtime > Change runtime type > GPU`
2. Upload `algerian_dishes_dataset.zip` to your Google Drive (MyDrive root).
3. Run cells top to bottom.

Outputs saved to your Drive: model checkpoint + confusion matrix.


## 1. Mount Drive and unzip the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/algerian_dishes_dataset.zip" /content/
!cd /content && unzip -q -o algerian_dishes_dataset.zip
!echo "classes:" && ls /content/data | wc -l

## 2. Imports and device check

In [ ]:
import copy
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
assert device.type == 'cuda', 'No GPU. Runtime > Change runtime type > GPU.'

## 3. Config

In [ ]:
DATA_DIR    = '/content/data'
IMG_SIZE    = 224
BATCH       = 32
HEAD_EPOCHS = 8     # phase 1: train classifier head only
FT_EPOCHS   = 12    # phase 2: fine-tune whole network
OUT_PATH    = '/content/drive/MyDrive/resnet50_algerian_dishes.pth'

## 4. Datasets, augmentation, stratified split

Train transforms are augmented (crop, flip, rotation, jitter) — 100 images per
class is small, so augmentation matters. Validation uses a plain resize/crop.
The 80/20 split is stratified so every class is balanced in both sets.


In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

base    = datasets.ImageFolder(DATA_DIR)
classes = base.classes
targets = [lbl for _, lbl in base.samples]
train_idx, val_idx = train_test_split(
    range(len(targets)), test_size=0.2, stratify=targets, random_state=42)

train_ds = Subset(datasets.ImageFolder(DATA_DIR, train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(DATA_DIR, eval_tf),  val_idx)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(len(classes), 'classes |', len(train_ds), 'train |', len(val_ds), 'val')

## 5. Model — ResNet-50 pretrained, backbone frozen

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)
criterion = nn.CrossEntropyLoss()

## 6. Train / eval loop

In [ ]:
def run_epoch(dl, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    loss_sum = correct = n = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(is_train):
            out  = model(x)
            loss = criterion(out, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        n        += x.size(0)
    return loss_sum / n, correct / n

## 7. Phase 1 — train the classifier head

In [ ]:
best_acc, best_wts = 0.0, copy.deepcopy(model.state_dict())
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
for e in range(HEAD_EPOCHS):
    _, ta = run_epoch(train_dl, opt)
    _, va = run_epoch(val_dl)
    print(f'[head {e+1:02d}/{HEAD_EPOCHS}]  train_acc {ta:.3f}  val_acc {va:.3f}')
    if va > best_acc:
        best_acc, best_wts = va, copy.deepcopy(model.state_dict())

## 8. Phase 2 — fine-tune the whole network

In [ ]:
for p in model.parameters():
    p.requires_grad = True
opt   = torch.optim.Adam(model.parameters(), lr=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
for e in range(FT_EPOCHS):
    _, ta = run_epoch(train_dl, opt)
    _, va = run_epoch(val_dl)
    sched.step()
    print(f'[fine {e+1:02d}/{FT_EPOCHS}]  train_acc {ta:.3f}  val_acc {va:.3f}')
    if va > best_acc:
        best_acc, best_wts = va, copy.deepcopy(model.state_dict())
model.load_state_dict(best_wts)
print(f'\nbest val accuracy: {best_acc:.3f}')

## 9. Evaluation — per-class report and confusion matrix

The confusion matrix tells you which dishes the model mixes up. Visually
similar pairs (chakhchoukha/rechta, makroud/kalb_el_louz) are the likely
offenders — that is what Day 4 iteration targets.


In [ ]:
model.eval()
preds, gts = [], []
with torch.no_grad():
    for x, y in val_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        gts   += y.tolist()

print(classification_report(gts, preds, target_names=classes, digits=3))

cm = confusion_matrix(gts, preds)
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=90)
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
plt.title('Confusion matrix (validation)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=120)
plt.show()

## 10. Save the model to Drive

In [ ]:
torch.save({
    'state_dict': best_wts,
    'classes':    classes,
    'img_size':   IMG_SIZE,
    'val_acc':    best_acc,
}, OUT_PATH)
print('saved ->', OUT_PATH)
print('keep this file — the Gradio app (Day 5) loads it.')